[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/freeforall1932-design/dionysus/blob/arena/5bf33a0c-dionysus/workbench/projects/kobold-cli/kobold-cli.ipynb)

# Kobold CLI — owner test notebook

This is the Dionysus project's small, coding-focused KoboldCpp notebook. It is **not** a copy of the official roleplay notebook.

## Safe first run

1. In Colab, choose **Runtime → Change runtime type → GPU**.
2. Run **Cell 1** exactly as delivered. The default mode is **Preflight only**: it checks the form, GPU, disk space, and prints the exact launch command. It does **not** download a model and does **not** open a public tunnel.
3. Read the preflight output. Only if you want the test download, change Cell 1 to **Download + verify only**, rerun Cell 1, then run Cell 2.
4. Only if you want the live endpoint, change Cell 1 to **Launch password-protected endpoint**, rerun Cell 1, then run Cells 2 and 3.

## Safety notes

- No silent-audio or other keep-alive trick is included.
- Launch mode generates a fresh API password and binds KoboldCpp to `127.0.0.1` inside the runtime; the Cloudflare tunnel is the public path.
- KoboldCpp prints its parsed arguments at startup, so the generated password may appear in the launch-cell output. Copy what you need, then clear the cell output. Do not save or share a copy with outputs.
- Stop the endpoint with the cell's stop/interrupt button. Closing the browser tab does not necessarily stop the Colab runtime immediately; use **Runtime → Disconnect and delete runtime** when finished.
- Colab terms and free-runtime limits are still an open project check (W-021/Q14). This notebook is an owner-directed test artifact, not proof that a usage pattern is permitted.
- The KoboldCpp release binary is pinned and SHA-256 verified. If the remote tunnel is used, KoboldCpp itself downloads Cloudflare's `cloudflared` helper from its moving latest-release URL; that upstream behaviour is outside this notebook's pin.


In [ ]:
#@title 1 | Configure and preflight (safe default) { display-mode: "form" }
#@markdown Run this cell first. **Preflight only** makes no downloads and opens no tunnel.

RunMode = "Preflight only (no downloads, no tunnel)" #@param ["Preflight only (no downloads, no tunnel)", "Download + verify only (no launch)", "Launch password-protected endpoint"]
ModelPreset = "Fast smoke test - Qwen2.5-Coder 1.5B Q4_K_M" #@param ["Fast smoke test - Qwen2.5-Coder 1.5B Q4_K_M", "Balanced coder - Qwen2.5-Coder 7B Q5_K_M", "Stronger coder - Qwen2.5-Coder 14B Q4_K_M", "Large tight fit - Mistral Small 3.2 24B Q4_K_S", "Custom URL only"]
ModelURL = "" #@param {type:"string"}
UsePresetRecommendedContext = True #@param {type:"boolean"}
ContextSize = 16384 #@param [8192, 16384, 32768, 65536] {allow-input: true}
KVCache = "q8_0" #@param ["f16", "bf16", "q8_0", "q5_1", "q4_0"]
FlashAttention = True #@param {type:"boolean"}
GPULayers = "Auto (-1)" #@param ["Auto (-1)", "999 (all)", "0 (CPU)"] {allow-input: true}
BatchSize = 512 #@param [128, 256, 512, 1024, 2048, 4096] {allow-input: true}
GenLimit = 0 #@param {type:"integer"}
ParallelRequests = 1 #@param [1, 2, 4] {allow-input: true}
Port = 5001 #@param {type:"integer"}
UseRemoteTunnel = True #@param {type:"boolean"}
UseMMQ = True #@param {type:"boolean"}
EnableWebSearch = False #@param {type:"boolean"}
DeleteExistingModels = False #@param {type:"boolean"}
ExtraArgs = "" #@param {type:"string"}

#@markdown Advanced RoPE: leave RopeScale at 0 for KoboldCpp automatic NTK-aware scaling.
RopeScale = 0.0 #@param {type:"number"}
RopeBase = 10000.0 #@param {type:"number"}

from pathlib import Path
import re
import shlex
import shutil
import subprocess
import urllib.parse

KOBOLDCPP_VERSION = "v1.123.1"
KOBOLDCPP_URL = (
    "https://github.com/LostRuins/koboldcpp/releases/download/"
    "v1.123.1/koboldcpp-linux-x64"
)
KOBOLDCPP_SHA256 = "f23cd1faacd96b5627ae009012e384f9b9b57f1a3e8e49b3c2fb10cd7fd6fc79"
KOBOLDCPP_BYTES = 648_375_216
WORKDIR = Path("/content/kobold-cli")
MODEL_DIR = WORKDIR / "models"
BIN_PATH = WORKDIR / f"koboldcpp-linux-x64-{KOBOLDCPP_VERSION}"

PRESETS = {
    "Fast smoke test - Qwen2.5-Coder 1.5B Q4_K_M": {
        "url": "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/f86cb2c1fa58255f8052cc32aeede1b7482d4361/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf",
        "recommended_context": 16384,
        "approx_gb": 1.0,
        "recommended_vram_gb": 4,
        "license_note": "Qwen model card: Apache-2.0",
    },
    "Balanced coder - Qwen2.5-Coder 7B Q5_K_M": {
        "url": "https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/13fb94bfda8c8cf22497dc57b78f391a9acb426a/qwen2.5-coder-7b-instruct-q5_k_m.gguf",
        "recommended_context": 32768,
        "approx_gb": 5.3,
        "recommended_vram_gb": 10,
        "license_note": "Qwen model card: Apache-2.0",
    },
    "Stronger coder - Qwen2.5-Coder 14B Q4_K_M": {
        "url": "https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/d0a692ef765eefbf2fabb130b3cb2e8917e3d225/qwen2.5-coder-14b-instruct-q4_k_m.gguf",
        "recommended_context": 32768,
        "approx_gb": 9.0,
        "recommended_vram_gb": 13,
        "license_note": "Qwen model card: Apache-2.0",
    },
    "Large tight fit - Mistral Small 3.2 24B Q4_K_S": {
        "url": "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/b750ec2299225e492f1bd27cab88a0a595fa848f/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf",
        "recommended_context": 16384,
        "approx_gb": 13.5,
        "recommended_vram_gb": 16,
        "license_note": "Model card: Apache-2.0",
    },
    "Custom URL only": {
        "url": "",
        "recommended_context": 16384,
        "approx_gb": None,
        "recommended_vram_gb": None,
        "license_note": "Check the model license yourself before use",
    },
}

if RunMode.startswith("Preflight"):
    RUN_MODE = "preflight"
elif RunMode.startswith("Download"):
    RUN_MODE = "download"
elif RunMode.startswith("Launch"):
    RUN_MODE = "launch"
else:
    raise ValueError(f"Unknown RunMode: {RunMode!r}")

if ModelPreset not in PRESETS:
    raise ValueError(f"Unknown ModelPreset: {ModelPreset!r}")
PRESET = PRESETS[ModelPreset]
MODEL_URL = ModelURL.strip() or PRESET["url"]
if not MODEL_URL:
    raise ValueError("Choose a preset or paste an HTTPS .gguf URL into ModelURL.")

parsed_model_url = urllib.parse.urlparse(MODEL_URL)
if parsed_model_url.scheme != "https" or not parsed_model_url.netloc:
    raise ValueError("ModelURL must be a full HTTPS URL.")
MODEL_FILENAME = Path(urllib.parse.unquote(parsed_model_url.path)).name
if not MODEL_FILENAME.lower().endswith(".gguf"):
    raise ValueError("ModelURL must point directly to one .gguf file.")
if re.search(r"-\d{5}-of-\d{5}\.gguf$", MODEL_FILENAME, re.IGNORECASE):
    raise ValueError(
        "That looks like one shard of a split GGUF. This first notebook downloads "
        "one complete .gguf only; choose the non-split file."
    )
MODEL_PATH = MODEL_DIR / MODEL_FILENAME

EFFECTIVE_CONTEXT = int(
    PRESET["recommended_context"]
    if UsePresetRecommendedContext and not ModelURL.strip()
    else ContextSize
)
if not 256 <= EFFECTIVE_CONTEXT <= 524288:
    raise ValueError("Context size must be between 256 and 524288.")
if KVCache not in {"f16", "bf16", "q8_0", "q5_1", "q4_0"}:
    raise ValueError(f"Unsupported KVCache value: {KVCache!r}")
if int(BatchSize) not in {-1, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096}:
    raise ValueError("BatchSize must be one of KoboldCpp allowed values: -1, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096.")
if int(GenLimit) < 0:
    raise ValueError("GenLimit cannot be negative. Use 0 for no notebook-level cap.")
if not 1 <= int(ParallelRequests) <= 32:
    raise ValueError("ParallelRequests must be between 1 and 32.")
if not 1024 <= int(Port) <= 65535:
    raise ValueError("Port must be between 1024 and 65535.")
if float(RopeScale) != 0.0 and not 0.0 < float(RopeScale) <= 10.0:
    raise ValueError("RopeScale must be 0 (automatic) or greater than 0 and no more than 10.")
if float(RopeScale) != 0.0 and float(RopeBase) <= 0:
    raise ValueError("RopeBase must be positive when RopeScale is set.")

def parse_gpu_layers(value):
    text = str(value).strip()
    if text.lower().startswith("auto"):
        return -1
    if text.startswith("999"):
        return 999
    if text.startswith("0"):
        return 0
    try:
        layers = int(text)
    except ValueError as exc:
        raise ValueError("GPULayers must be Auto, 999, 0, or an integer layer count.") from exc
    if layers < -1:
        raise ValueError("GPULayers cannot be below -1.")
    return layers

GPU_LAYERS = parse_gpu_layers(GPULayers)

PROTECTED_EXTRA_ARGS = {
    "--admin", "--admindir", "--adminpassword", "--benchmark", "--config",
    "--contextsize", "--ctx-size", "--downloaddir", "--gpulayers", "--host",
    "--model", "--password", "--port", "--prompt", "--quantkv",
    "--remotetunnel", "--ropeconfig", "--showgui", "--skiplauncher",
    "--usecublas", "--usecuda", "--usehipblas",
}

def parse_extra_args(text):
    if not text.strip():
        return []
    try:
        tokens = shlex.split(text)
    except ValueError as exc:
        raise ValueError(f"ExtraArgs could not be parsed: {exc}") from exc
    if not tokens[0].startswith("-"):
        raise ValueError("ExtraArgs must contain only KoboldCpp flags, starting with '-'.")
    for token in tokens:
        flag = token.split("=", 1)[0]
        if flag in PROTECTED_EXTRA_ARGS:
            raise ValueError(
                f"ExtraArgs may not override protected notebook setting {flag!r}. "
                "Change the matching form field instead."
            )
    return tokens

EXTRA_ARGS = parse_extra_args(ExtraArgs)

def build_launch_command():
    command = [
        str(BIN_PATH),
        "--model", str(MODEL_PATH),
        "--usecuda", "0", "mmq" if UseMMQ else "nommq",
        "--gpulayers", str(GPU_LAYERS),
        "--contextsize", str(EFFECTIVE_CONTEXT),
        "--quantkv", KVCache,
        "--blasbatchsize", str(int(BatchSize)),
        "--host", "127.0.0.1",
        "--port", str(int(Port)),
        "--parallelrequests", str(int(ParallelRequests)),
        "--skiplauncher",
    ]
    if not FlashAttention:
        command.append("--noflashattention")
    if int(GenLimit) > 0:
        command.extend(["--genlimit", str(int(GenLimit))])
    if float(RopeScale) != 0.0:
        command.extend(["--ropeconfig", str(float(RopeScale)), str(float(RopeBase))])
    if EnableWebSearch:
        command.append("--websearch")
    command.append("--quiet")
    if UseRemoteTunnel:
        command.append("--remotetunnel")
    command.extend(EXTRA_ARGS)
    return command

def gpu_summary():
    exe = shutil.which("nvidia-smi")
    if not exe:
        return None, "nvidia-smi not found (no NVIDIA GPU runtime detected)"
    try:
        result = subprocess.run(
            [exe, "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            text=True, capture_output=True, timeout=15, check=False,
        )
    except Exception as exc:
        return None, f"nvidia-smi failed: {type(exc).__name__}: {exc}"
    if result.returncode != 0:
        return None, (result.stderr or result.stdout or "nvidia-smi failed").strip()
    lines = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    return lines, None

gpu_lines, gpu_error = gpu_summary()
disk_root = WORKDIR if WORKDIR.parent.exists() else Path.cwd()
disk = shutil.disk_usage(disk_root)
approx_model_gb = PRESET["approx_gb"] if not ModelURL.strip() else None
approx_download_gb = None
if approx_model_gb is not None:
    approx_download_gb = approx_model_gb + KOBOLDCPP_BYTES / 1_000_000_000

print("=" * 72)
print("Kobold CLI preflight")
print("=" * 72)
print(f"Mode: {RunMode}")
print(f"KoboldCpp: {KOBOLDCPP_VERSION} (pinned; SHA-256 checked in Cell 2)")
print(f"Model source: {'custom URL' if ModelURL.strip() else ModelPreset}")
print(f"Model URL: {MODEL_URL}")
print(f"Model file: {MODEL_PATH}")
print(f"License note: {PRESET['license_note'] if not ModelURL.strip() else 'Custom model: check its license yourself.'}")
print(
    "Settings: "
    f"context={EFFECTIVE_CONTEXT}, quantkv={KVCache}, flash_attention={FlashAttention}, "
    f"gpu_layers={GPU_LAYERS}, batch={int(BatchSize)}, genlimit={int(GenLimit)}, "
    f"parallel={int(ParallelRequests)}, mmq={UseMMQ}, tunnel={UseRemoteTunnel}"
)
if gpu_lines:
    for line in gpu_lines:
        print(f"GPU (name,total MiB,free MiB): {line}")
else:
    print(f"GPU: not confirmed - {gpu_error}")
print(f"Disk free at {disk_root}: {disk.free / 1_000_000_000:.1f} GB")
if approx_download_gb is not None:
    print(f"Approx first download: {approx_download_gb:.1f} GB plus tunnel helper/model overhead")
if PRESET["recommended_vram_gb"] and not ModelURL.strip():
    print(f"Preset planning VRAM: about {PRESET['recommended_vram_gb']} GB or more (not a guarantee)")

warnings = []
if RUN_MODE == "launch" and not gpu_lines:
    warnings.append("Launch mode needs a Colab GPU runtime; Cell 3 will stop if no GPU is confirmed.")
if not FlashAttention and KVCache not in {"f16", "bf16"}:
    warnings.append("KV quantization without flash attention is not recommended by KoboldCpp; only K cache gets the full treatment.")
if EFFECTIVE_CONTEXT > 65536:
    warnings.append("Very large context can exhaust VRAM or slow prompt processing sharply, even with q8_0/q4_0 KV cache.")
if RUN_MODE == "launch" and not UseRemoteTunnel:
    warnings.append("Remote tunnel is off: the endpoint will only be reachable inside this Colab runtime.")
if DeleteExistingModels:
    warnings.append("DeleteExistingModels is ON: Cell 2 will delete .gguf files only inside /content/kobold-cli/models.")
if approx_download_gb is not None and disk.free < int((approx_download_gb + 1.0) * 1_000_000_000):
    warnings.append("Free disk may be too small for the binary plus selected model.")

if warnings:
    print("\nWarnings:")
    for warning in warnings:
        print(f"- {warning}")

print("\nExact launch command (password is supplied by environment, not in argv):")
print(shlex.join(build_launch_command()))
print("\nPreflight passed. No download or launch happened in this cell.")
if RUN_MODE == "preflight":
    print("Next: change RunMode only when you are ready, rerun this cell, then run Cell 2.")
elif RUN_MODE == "download":
    print("Next: run Cell 2. It will download and verify, but will not launch.")
else:
    print("Next: run Cell 2, then Cell 3. Cell 3 stays busy while the server is running.")


In [ ]:
#@title 2 | Download and verify (does not launch) { display-mode: "form" }
#@markdown This cell is a no-op in Preflight mode. Downloads are resumable `.part` files under `/content/kobold-cli`.

from pathlib import Path
import hashlib
import os
import time
import urllib.error
import urllib.request

try:
    _mode = RUN_MODE
    _bin_path = BIN_PATH
    _model_path = MODEL_PATH
except NameError as exc:
    raise RuntimeError("Run Cell 1 first so the configuration exists.") from exc

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verify_gguf(path):
    if not path.is_file() or path.stat().st_size < 1024 * 1024:
        return False
    with open(path, "rb") as handle:
        return handle.read(4) == b"GGUF"

def download_with_resume(url, destination, label, expected_sha256=None, require_gguf=False):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        if expected_sha256:
            actual = sha256_file(destination)
            if actual == expected_sha256:
                print(f"{label}: already downloaded and SHA-256 verified ({destination.stat().st_size:,} bytes)")
                return destination
            print(f"{label}: existing file failed SHA-256; downloading again")
            destination.unlink()
        elif require_gguf and verify_gguf(destination):
            print(f"{label}: already downloaded and GGUF header verified ({destination.stat().st_size:,} bytes)")
            return destination
        elif not require_gguf:
            print(f"{label}: already present ({destination.stat().st_size:,} bytes)")
            return destination
        else:
            print(f"{label}: existing file is not a valid-looking GGUF; downloading again")
            destination.unlink()

    part = destination.with_name(destination.name + ".part")
    sidecar = destination.with_name(destination.name + ".part.url")
    if sidecar.is_file() and sidecar.read_text(encoding="utf-8").strip() != url:
        print(f"{label}: partial file belongs to a different URL; discarding it")
        part.unlink(missing_ok=True)
    sidecar.write_text(url + "\n", encoding="utf-8")

    for attempt in range(1, 4):
        existing = part.stat().st_size if part.is_file() else 0
        headers = {"User-Agent": "dionysus-kobold-cli/1.0"}
        if existing:
            headers["Range"] = f"bytes={existing}-"
        request = urllib.request.Request(url, headers=headers)
        try:
            with urllib.request.urlopen(request, timeout=90) as response:
                status = getattr(response, "status", 200)
                if existing and status == 200:
                    existing = 0
                mode = "ab" if existing and status == 206 else "wb"
                length = response.headers.get("Content-Length")
                total = existing + int(length) if length and status == 206 else int(length) if length else None
                downloaded = existing
                next_report = downloaded + 256 * 1024 * 1024
                print(
                    f"{label}: {'resuming' if existing else 'downloading'} "
                    f"from {existing:,} bytes" + (f" of about {total:,}" if total else "")
                )
                with open(part, mode) as output:
                    while True:
                        chunk = response.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        output.write(chunk)
                        downloaded += len(chunk)
                        if downloaded >= next_report:
                            print(f"{label}: {downloaded:,} bytes downloaded")
                            next_report = downloaded + 256 * 1024 * 1024
                    output.flush()
                    os.fsync(output.fileno())
            break
        except urllib.error.HTTPError as exc:
            if exc.code == 416 and part.is_file() and part.stat().st_size:
                print(f"{label}: server says the partial file is already complete; verifying it")
                break
            if attempt == 3:
                raise
            print(f"{label}: HTTP {exc.code}; retrying in 3 seconds...")
            time.sleep(3)
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            if attempt == 3:
                raise
            print(f"{label}: download interrupted ({type(exc).__name__}: {exc}); retrying in 3 seconds...")
            time.sleep(3)

    if expected_sha256:
        actual = sha256_file(part)
        if actual != expected_sha256:
            part.unlink(missing_ok=True)
            sidecar.unlink(missing_ok=True)
            raise RuntimeError(
                f"{label}: SHA-256 mismatch. Expected {expected_sha256}, got {actual}. "
                "The bad download was deleted and nothing was launched."
            )
        print(f"{label}: SHA-256 verified")
    if require_gguf and not verify_gguf(part):
        size = part.stat().st_size if part.is_file() else 0
        part.unlink(missing_ok=True)
        sidecar.unlink(missing_ok=True)
        raise RuntimeError(f"{label}: downloaded file is not a valid-looking GGUF (size {size:,}). It was deleted.")

    part.replace(destination)
    sidecar.unlink(missing_ok=True)
    print(f"{label}: ready at {destination} ({destination.stat().st_size:,} bytes)")
    return destination

def ensure_downloads():
    WORKDIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    if DeleteExistingModels:
        for old in MODEL_DIR.glob("*.gguf"):
            print(f"Deleting cached model: {old}")
            old.unlink()
        for old in MODEL_DIR.glob("*.gguf.part"):
            print(f"Deleting partial model: {old}")
            old.unlink()
    download_with_resume(
        KOBOLDCPP_URL,
        BIN_PATH,
        f"KoboldCpp {KOBOLDCPP_VERSION}",
        expected_sha256=KOBOLDCPP_SHA256,
        require_gguf=False,
    )
    BIN_PATH.chmod(0o755)
    download_with_resume(
        MODEL_URL,
        MODEL_PATH,
        "Model",
        expected_sha256=None,
        require_gguf=True,
    )
    print("Downloads verified. Nothing has been launched by Cell 2.")

if RUN_MODE == "preflight":
    print("Preflight mode: Cell 2 skipped. No files were downloaded.")
else:
    ensure_downloads()


In [ ]:
#@title 3 | Launch endpoint (only in Launch mode) { display-mode: "form" }
#@markdown This cell stays busy while KoboldCpp is running. Use the stop/interrupt button to stop it.

import os
import secrets
import shlex
import shutil
import subprocess

try:
    _mode = RUN_MODE
    _ensure = ensure_downloads
except NameError as exc:
    raise RuntimeError("Run Cell 1 and Cell 2 first.") from exc

if RUN_MODE != "launch":
    print(f"Mode is {RUN_MODE}: Cell 3 skipped. No server or tunnel was started.")
else:
    if not shutil.which("nvidia-smi"):
        raise RuntimeError(
            "No NVIDIA GPU was detected. In Colab choose Runtime -> Change runtime type -> GPU, "
            "then rerun Cells 1, 2, and 3."
        )
    ensure_downloads()

    API_PASSWORD = secrets.token_urlsafe(32)
    env = os.environ.copy()
    env["KCPP_PASSWORD"] = API_PASSWORD
    # Admin mode itself remains disabled. The same generated secret is also
    # supplied as the admin password so enabling admin functions later cannot
    # silently start unprotected.
    env["KCPP_ADMINPASSWORD"] = API_PASSWORD

    command = build_launch_command()
    print("=" * 72)
    print("Starting KoboldCpp. Wait for the model to load and for KoboldCpp to print the tunnel URL.")
    print("OpenAI-compatible base URL will be: <printed tunnel URL>/v1")
    print(f"API password / Bearer token: {API_PASSWORD}")
    print("Copy the tunnel URL and password, then clear this cell output. KoboldCpp may also print its parsed args.")
    print("Command:", shlex.join(command))
    print("=" * 72, flush=True)

    try:
        completed = subprocess.run(command, cwd=str(WORKDIR), env=env, check=False)
        return_code = completed.returncode
    except KeyboardInterrupt:
        print("\nStopped by user (KeyboardInterrupt).")
        return_code = 130

    if return_code not in (0, 130):
        raise RuntimeError(
            f"KoboldCpp exited with code {return_code}. Read the log above; common causes are "
            "out-of-VRAM (lower context or model size), an unsupported model/backend, or tunnel failure."
        )
    print("KoboldCpp stopped.")


## After the test

- Record the GPU name/VRAM, model preset, context, KV cache, whether flash attention stayed enabled, load time, and any error text.
- Test the OpenAI-compatible base URL (`<tunnel>/v1`) with the generated Bearer password.
- Do **not** paste the password or saved cell outputs into Git.
- Stop Cell 3, then use **Runtime → Disconnect and delete runtime**.
